# Lab #6: Keras MLP for Regression


## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

## 2. Dataset Preparation

### 2.1 Load and Explore the Dataset

In [ ]:
data = load_diabetes(as_frame=True)
df = data.frame

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
df.describe()

In [ ]:
print("Missing values per column:")
print(df.isna().sum())

**Observation:** The dataset has **no missing values** and **no categorical variables** — all 10 features are continuous numeric measurements, and the target (`target`) is a continuous variable (disease progression score). This confirms it is a clean, well-posed regression problem.

### 2.2 Identify Input Features and Target Variable

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

print("Input features:", list(X.columns))
print("Target variable: 'target' (continuous disease progression score)")
print("Target range:", y.min(), "-", y.max())

### 2.3 Handle Missing Values and Categorical Variables
As confirmed above, there are no missing values and no categorical columns, so no imputation or encoding is required.

### 2.4 Feature Scaling / Normalization
MLPs are sensitive to feature scale — unscaled inputs can cause unstable gradients and slow/unequal convergence, especially with **sigmoid** and **tanh** activations, which saturate for large input magnitudes. We apply `StandardScaler` (zero mean, unit variance) to both the input features and the target variable. Scaling the target is important here because Sigmoid/Tanh-based hidden layers otherwise struggle to propagate gradients toward the (0–346) scale of the raw target, and it lets us compare loss values fairly across activation functions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

x_scaler = StandardScaler()
X_train_s = x_scaler.fit_transform(X_train)
X_test_s = x_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test_s = y_scaler.transform(y_test.values.reshape(-1, 1)).flatten()

print("Train shape:", X_train_s.shape, " Test shape:", X_test_s.shape)

## 3. MLP Model Development

Architecture (kept constant across all experiments so comparisons are fair):
- **Input layer:** 10 features
- **Hidden layer 1:** 64 neurons
- **Hidden layer 2:** 32 neurons
- **Hidden layer 3:** 16 neurons *(a third hidden layer, beyond the required minimum of two, since this is a fairly small tabular dataset and extra depth doesn't cost much)*
- **Output layer:** 1 neuron, **linear activation** (required for regression, so the output isn't artificially bounded)

The activation function of the **hidden layers** is the variable we swap between experiments.

In [ ]:
def build_model(activation="relu", n_features=X_train_s.shape[1]):
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation=activation),
        layers.Dense(32, activation=activation),
        layers.Dense(16, activation=activation),
        layers.Dense(1)  # linear output neuron for regression
    ])
    return model

build_model("relu").summary()

## 4. Training / Evaluation Helper

To keep every experiment consistent, one function builds the model, compiles it with the given activation/loss combination, trains with early stopping (to avoid over/under-training bias between configs), and returns evaluation metrics + the loss history for plotting.

**Fixed hyperparameters across all experiments:** Adam optimizer (lr=0.01), batch size = 16, up to 150 epochs with early stopping (patience=15 on validation loss, restoring best weights), 80/20 train/validation split within the training set.

In [ ]:
def train_and_eval(activation, loss_fn, tag, epochs=150, batch_size=16):
    model = build_model(activation)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
                   loss=loss_fn, metrics=["mae"])

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=15, restore_best_weights=True)

    history = model.fit(
        X_train_s, y_train_s,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=[early_stop]
    )

    y_pred_s = model.predict(X_test_s, verbose=0).flatten()
    y_pred = y_scaler.inverse_transform(y_pred_s.reshape(-1, 1)).flatten()
    y_true = y_test.values

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    plt.figure(figsize=(6, 4))
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.title(f"{tag}: Training vs Validation Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"{tag} | epochs trained: {len(history.history['loss'])} | "
          f"MAE: {mae:.3f}  RMSE: {rmse:.3f}  R2: {r2:.4f}")

    return {"tag": tag, "activation": activation, "loss_fn": loss_fn,
            "mse": mse, "rmse": rmse, "mae": mae, "r2": r2,
            "epochs_trained": len(history.history["loss"])}

results = []

## 5. Experiment 1 — Comparing Activation Functions

**Loss function held constant at MSE** while the hidden-layer activation varies across **ReLU**, **Sigmoid**, and **Tanh**.

**Why these three?**
- **ReLU** — computationally cheap, avoids vanishing gradients for positive inputs, generally the strongest default for tabular MLPs.
- **Sigmoid** — included as a classical baseline; saturates at both extremes, which is known to slow convergence in deep(ish) hidden stacks.
- **Tanh** — zero-centered version of sigmoid, usually converges better than sigmoid but can still saturate for large activations.

In [ ]:
res = train_and_eval("relu", "mse", "ActExp_relu")
results.append(res)

In [ ]:
res = train_and_eval("sigmoid", "mse", "ActExp_sigmoid")
results.append(res)

In [ ]:
res = train_and_eval("tanh", "mse", "ActExp_tanh")
results.append(res)

In [ ]:
best_act = max(results, key=lambda r: r["r2"])["activation"]
print("Best-performing activation function so far:", best_act)

**Interpretation — Activation Functions:**
- **ReLU** and **Tanh** produced very similar, and the best, R² (~0.409), clearly ahead of **Sigmoid** (R² ≈ 0.383).
- **Sigmoid** converged in the fewest epochs (18) but to a *worse* solution — a classic sign of early saturation flattening its gradients before the model finds a good fit, rather than genuinely fast, good convergence.
- ReLU is chosen as the activation function to carry forward into the loss-function experiment, since it gave the best R² and is also the cheapest to compute per step.

## 6. Experiment 2 — Comparing Loss Functions

**Activation held constant at ReLU** (the winner above) while the loss function varies across **MSE**, **MAE**, and **Huber**.

**Why these three, and why they suit this dataset:**
- **MSE (Mean Squared Error)** — the standard regression loss; penalizes large errors quadratically, which is appropriate if we want the model to strongly avoid big misses, but makes it sensitive to outliers.
- **MAE (Mean Absolute Error)** — penalizes all errors linearly, so it's more robust to outliers. The diabetes progression target has a fairly wide, slightly skewed range (25–346), so an outlier-robust loss is worth testing.
- **Huber Loss** — a hybrid: quadratic for small errors, linear for large ones. It's a natural middle ground between MSE's sensitivity and MAE's robustness, and is often recommended precisely for datasets — like this one — where some outlier patients exist but most errors are moderate.

In [ ]:
res = train_and_eval("relu", "mse", "LossExp_mse")
results.append(res)

In [ ]:
res = train_and_eval("relu", "mae", "LossExp_mae")
results.append(res)

In [ ]:
res = train_and_eval("relu", "huber", "LossExp_huber")
results.append(res)

**Interpretation — Loss Functions:**
- **MAE** produced the best overall test performance (R² = 0.4855, lowest MAE and RMSE), followed closely by **Huber** (R² = 0.4713). Plain **MSE** was clearly the weakest of the three (R² = 0.3969).
- This matches the outlier-robustness argument: the diabetes target has a right-skewed spread, and MSE's quadratic penalty lets a handful of hard-to-predict patients dominate the gradient, pulling the fit away from the bulk of the data. MAE and Huber down-weight those large residuals, giving a better overall fit.
- MAE also trained for the most epochs (40) before early stopping triggered — its gradient (constant magnitude, not shrinking near convergence) gives a steadier, if slightly noisier, training signal, so it kept improving validation loss for longer.

## 7. Comparison Table

In [ ]:
summary_df = pd.DataFrame([
    {"Experiment": r["tag"], "Activation": r["activation"], "Loss Function": r["loss_fn"],
     "MAE": round(r["mae"], 3), "RMSE": round(r["rmse"], 3), "R2": round(r["r2"], 4),
     "Epochs Trained": r["epochs_trained"]}
    for r in results
])
summary_df

| Experiment | Activation | Loss Function | MAE | RMSE | R² | Epochs Trained |
|---|---|---|---|---|---|---|
| Model 1 | ReLU | MSE | 45.508 | 55.944 | 0.4093 | 34 |
| Model 2 | Sigmoid | MSE | 45.096 | 57.176 | 0.3830 | 18 |
| Model 3 | Tanh | MSE | 45.797 | 55.979 | 0.4085 | 20 |
| Model 4 (loss sweep, ReLU fixed) | ReLU | MSE | 44.820 | 56.525 | 0.3969 | 29 |
| Model 5 (loss sweep, ReLU fixed) | ReLU | MAE | **40.348** | **52.208** | **0.4855** | 40 |
| Model 6 (loss sweep, ReLU fixed) | ReLU | Huber | 42.643 | 52.924 | 0.4713 | 29 |

*(Small differences between "Model 1" and "Model 4" — both ReLU + MSE — come from the randomness of dropout-free weight initialization and mini-batch ordering across separate training runs; both are consistent with each other.)*

## 8. Final Model Selection

In [ ]:
best = max(results, key=lambda r: r["r2"])
print(f"Best combination: activation='{best['activation']}', loss='{best['loss_fn']}'")
print(f"Test R²: {best['r2']:.4f}  RMSE: {best['rmse']:.3f}  MAE: {best['mae']:.3f}")

In [ ]:
# --- Train and store the FINAL model (ReLU + MAE, the winning combo) globally ---
model = build_model("relu")
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss="mae", metrics=["mae"])

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
model.fit(X_train_s, y_train_s, validation_split=0.2, epochs=150, batch_size=16,
          verbose=0, callbacks=[early_stop])

print("Final model trained and ready for inference.")

**Final model: ReLU activation + MAE loss.** It gave the best R², the lowest RMSE, and the lowest MAE on the held-out test set among all six configurations tested.

## 9. Analysis and Interpretation

**Which activation function performed best?**
ReLU and Tanh performed almost identically and both clearly beat Sigmoid on this architecture. ReLU is preferred going forward for its lower computational cost and equally strong accuracy.

**Which loss function performed best?**
MAE, by a clear margin over Huber and MSE, on both R² and RMSE.

**Which combination produced the best regression performance?**
ReLU (hidden layers) + MAE (loss) — R² = 0.4855, the best of all six configurations tested.

**Did the choice of activation function significantly affect convergence?**
Yes, but more in *training dynamics* than in accuracy: Sigmoid converged (i.e., early-stopping triggered) after only 18 epochs — noticeably faster than ReLU (34) or Tanh (20) — but converged to a **worse** solution, consistent with its gradients saturating and shrinking early. ReLU and Tanh both kept improving for longer and reached better minima.

**Did any loss function make the model more robust to outliers?**
Yes — MAE and Huber both outperformed plain MSE, supporting the expectation that this dataset (with a moderately skewed target distribution) benefits from a loss that doesn't let a few large residuals dominate training. MAE gave the strongest robustness benefit here; Huber landed in between MAE and MSE, as expected from its hybrid design.

**Overfitting / underfitting from the learning curves?**
Across all six loss curve plots, training and validation loss track closely together and flatten out rather than diverging — there is no visible overfitting gap. If anything, the R² values (~0.38–0.49) suggest mild **underfitting**: the model is not overfit, but a small 10-feature, 442-sample dataset caps how much variance any MLP can explain. This is consistent with published benchmarks on this dataset, where even well-tuned models typically reach R² in the 0.4–0.5 range without extensive feature engineering.

### Overall Conclusion
For this diabetes-progression regression task, a 3-hidden-layer MLP with **ReLU activations** and **MAE loss** gave the best generalization on unseen data. The activation choice mattered mainly for training stability and convergence quality (Sigmoid underperforming due to saturation), while the loss choice mattered for final accuracy — reflecting the dataset's sensitivity to outlier-driven residuals under a squared-error objective.

Practical application : patient risk scoring

In [ ]:
import pandas as pd

def predict_patient_risk(model, patient_features, x_scaler, y_scaler):
    """
    patient_features: dict of the 10 feature values for one patient
    Returns: predicted disease-progression risk score (higher = higher risk)
    """
    feature_order = ['age','sex','bmi','bp','s1','s2','s3','s4','s5','s6']
    x_df = pd.DataFrame([patient_features], columns=feature_order)
    x_scaled = x_scaler.transform(x_df)

    pred_scaled = model.predict(x_scaled, verbose=0).flatten()
    pred_risk = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()[0]
    return pred_risk

# Dummy patient — plausible values in this dataset's normalized units
# (sklearn's diabetes dataset ships with features already mean-centered/scaled,
#  so these small floats represent a roughly average-to-slightly-elevated patient)
dummy_patient = {
    'age': 0.038, 'sex': 0.050, 'bmi': 0.061, 'bp': 0.021,
    's1': -0.044, 's2': -0.035, 's3': -0.043, 's4': -0.002,
    's5': 0.019, 's6': -0.017
}

risk_score = predict_patient_risk(model, dummy_patient, x_scaler, y_scaler)
print(f"Predicted disease progression risk score: {risk_score:.1f}")

if risk_score > 200:
    print("Triage flag: HIGH risk — recommend priority follow-up")
elif risk_score > 120:
    print("Triage flag: MODERATE risk — routine follow-up")
else:
    print("Triage flag: LOW risk")